# Traffic Forecaster (LightGBM)
Predicts highway speeds 15/30/60 minutes ahead from sensor data.

**How to run:** Runtime ▸ Run all. By default it creates the demo dataset; to use a real dataset, upload your CSV in step 2.

## 1. Install libraries

In [ ]:
!pip install -q lightgbm pandas scikit-learn

## 2. Dataset
Either the demo dataset is generated, or upload your own CSV (same columns).

In [ ]:
"""Generate a synthetic METR-LA-style traffic speed dataset (wide format).

Columns: timestamp, s00 ... sNN  (speed in mph, one row per 5 minutes).
This is the SAME layout you get from METR-LA / PeMS after converting to CSV,
so you can later swap in the real file without changing train.py.
"""
import numpy as np
import pandas as pd


def generate(path="data/traffic_speeds.csv", n_sensors=20, days=28, seed=42):
    rng = np.random.default_rng(seed)
    idx = pd.date_range("2024-03-04", periods=days * 288, freq="5min")
    hour = idx.hour.values + idx.minute.values / 60
    weekday = (idx.dayofweek.values < 5).astype(float)

    base = rng.uniform(62, 70, n_sensors)            # free-flow speed
    am_sev = rng.uniform(0.15, 0.45, n_sensors)      # morning rush severity
    pm_sev = rng.uniform(0.15, 0.50, n_sensors)      # evening rush severity
    am = np.exp(-((hour - 8.0) / 1.0) ** 2)
    pm = np.exp(-((hour - 17.3) / 1.3) ** 2)
    midday = 0.25 * np.exp(-((hour - 13) / 2.5) ** 2)

    speeds = np.zeros((len(idx), n_sensors))
    for s in range(n_sensors):
        drop = weekday * (am_sev[s] * am + pm_sev[s] * pm) + (1 - weekday) * midday * am_sev[s]
        speeds[:, s] = base[s] * (1 - drop)

    # random incidents that propagate to upstream sensors with a delay
    for _ in range(days * 3):
        start = rng.integers(0, len(idx) - 60)
        dur = rng.integers(6, 18)
        s0 = rng.integers(0, n_sensors)
        mag = rng.uniform(0.25, 0.55)
        for k in range(4):
            s = s0 - k
            if s < 0:
                break
            a = start + k * 2
            b = min(a + dur + k * 3, len(idx))
            speeds[a:b, s] *= 1 - mag * (0.8 ** k)

    noise = pd.DataFrame(rng.normal(0, 1, speeds.shape)).ewm(alpha=0.4).mean().values * 2.5
    speeds = np.clip(speeds + noise, 5, 75).round(1)

    df = pd.DataFrame(speeds, index=idx, columns=[f"s{i:02d}" for i in range(n_sensors)])
    df.index.name = "timestamp"
    df.to_csv(path)
    return df




In [ ]:
import os
os.makedirs("data", exist_ok=True); os.makedirs("outputs", exist_ok=True)
CSV = "data/traffic_speeds.csv"

# --- OPTION A (default): generate the demo dataset ---
generate(CSV)

# --- OPTION B: upload your own CSV (uncomment) ---
# from google.colab import files
# up = files.upload()              # pick your CSV
# CSV = list(up.keys())[0]

import pandas as pd
pd.read_csv(CSV).head()

## 3. Training code

In [ ]:
"""Traffic speed forecaster: predicts speed 15 / 30 / 60 min ahead for every sensor.

Model : LightGBM (falls back to sklearn HistGradientBoosting if LightGBM is missing)
Input : wide CSV  ->  timestamp, <sensor_1>, <sensor_2>, ...  (5-minute data)
Run   : python train.py --csv data/traffic_speeds.csv
"""
import argparse
import json
import os

import numpy as np
import pandas as pd
from sklearn.metrics import (accuracy_score, f1_score, mean_absolute_error,
                             mean_squared_error, precision_score, r2_score,
                             recall_score)

STEP_MIN = 5
HORIZONS = {"15min": 3, "30min": 6, "60min": 12}   # in 5-min steps
LAGS = [0, 1, 2, 3, 6, 12]
CONGESTION_MPH = 45                                  # "congested" if speed < 45 mph


def make_regressor():
    try:
        from lightgbm import LGBMRegressor
        return LGBMRegressor(n_estimators=400, learning_rate=0.05, num_leaves=63,
                             subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                             random_state=42, verbose=-1), "LightGBM"
    except ImportError:
        from sklearn.ensemble import HistGradientBoostingRegressor
        return HistGradientBoostingRegressor(max_iter=300, learning_rate=0.08,
                                             random_state=42), "sklearn HistGradientBoosting (fallback)"


def build_features(df, horizon):
    sensors = list(df.columns)
    net_mean = df.mean(axis=1)
    h = df.index.hour + df.index.minute / 60
    frames = []
    for i, s in enumerate(sensors):
        x = pd.DataFrame(index=df.index)
        for k in LAGS:
            x[f"lag_{k}"] = df[s].shift(k)
        x["roll_mean_6"] = df[s].rolling(6).mean()
        x["roll_std_6"] = df[s].rolling(6).std()
        up = df[sensors[i - 1]] if i > 0 else df[s]
        down = df[sensors[i + 1]] if i < len(sensors) - 1 else df[s]
        x["upstream"], x["downstream"] = up, down
        x["upstream_lag3"] = up.shift(3)
        x["network_mean"] = net_mean
        x["hour_sin"] = np.sin(2 * np.pi * h / 24)
        x["hour_cos"] = np.cos(2 * np.pi * h / 24)
        x["dow"] = df.index.dayofweek
        x["sensor_id"] = i
        x["current"] = df[s]
        x["target"] = df[s].shift(-horizon)
        x["timestamp"] = df.index
        frames.append(x)
    return pd.concat(frames).dropna()


def main(csv_path="data/traffic_speeds.csv", out_dir="outputs"):
    os.makedirs(out_dir, exist_ok=True)
    df = pd.read_csv(csv_path, parse_dates=["timestamp"]).set_index("timestamp").sort_index()
    cutoff = df.index[int(len(df) * 0.75)]            # time-based split (no leakage)
    results = {}
    for name, steps in HORIZONS.items():
        data = build_features(df, steps)
        train = data[data.timestamp <= cutoff - pd.Timedelta(minutes=steps * STEP_MIN)]
        test = data[data.timestamp > cutoff]
        feats = [c for c in data.columns if c not in ("target", "current", "timestamp")]
        model, model_name = make_regressor()
        model.fit(train[feats], train["target"])
        pred = np.clip(model.predict(test[feats]), 0, 85)
        y = test["target"].values

        yc, pc = y < CONGESTION_MPH, pred < CONGESTION_MPH
        base_mae = mean_absolute_error(y, test["current"])   # "speed stays the same" baseline
        results[name] = {
            "MAE_mph": round(mean_absolute_error(y, pred), 3),
            "RMSE_mph": round(float(np.sqrt(mean_squared_error(y, pred))), 3),
            "R2": round(r2_score(y, pred), 4),
            "baseline_persistence_MAE_mph": round(base_mae, 3),
            "congestion_accuracy": round(accuracy_score(yc, pc), 4),
            "congestion_precision": round(precision_score(yc, pc, zero_division=0), 4),
            "congestion_recall": round(recall_score(yc, pc, zero_division=0), 4),
            "congestion_f1": round(f1_score(yc, pc, zero_division=0), 4),
            "congested_share_of_test": round(float(yc.mean()), 4),
            "train_rows": len(train), "test_rows": len(test),
        }
    results["_model"] = model_name
    with open(os.path.join(out_dir, "metrics.json"), "w") as f:
        json.dump(results, f, indent=2)
    print(f"Model: {model_name}   |   congestion = speed < {CONGESTION_MPH} mph")
    print(pd.DataFrame({k: v for k, v in results.items() if k != "_model"}).T.to_string())
    return results




## 4. Train + evaluate (prints accuracy / precision / recall / F1)

In [ ]:
results = main(CSV, 'outputs')

## Download your outputs

In [ ]:
from google.colab import files
import shutil
shutil.make_archive("outputs", "zip", "outputs")
files.download("outputs.zip")